<a href="https://colab.research.google.com/github/didarcevik/project/blob/master/CHATBOT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install \
    pandas \
    langchain \
    langchain-community \
    langchain-huggingface \
    huggingface-hub \
    faiss-cpu \
    sentence-transformers

In [ ]:
import os
from getpass import getpass

import pandas as pd
from google.colab import files
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint


In [ ]:
os.environ['HF_TOKEN'] = getpass('Enter your Hugging Face Hub token:')

In [ ]:
from google.colab import files

uploaded = files.upload()

import pandas as pd

df = pd.read_csv(list(uploaded.keys())[0])

data = df['instruction'].tolist() + df['response'].tolist()

In [ ]:
embeddings = HuggingFaceEmbeddings()
db = FAISS.from_texts(data, embeddings)

In [ ]:
llm_mistral = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.3",  # Correct Mistral model ID
    temperature=0.5,  # Set temperature explicitly
    max_length=128    # Set max_length explicitly
)

In [ ]:
mistral_qa = RetrievalQA.from_chain_type(llm=llm_mistral, chain_type="stuff", retriever=db.as_retriever())

In [ ]:
def chat():
    while True:
        user_input = input("You: ")
        response = mistral_qa.invoke(user_input)
        print("Bot:", response)
        if user_input.lower() == 'exit':
            break

chat()